# Day 033 — Exercise 4: process_batch

**What you'll build:** `process_batch(items, async_fn, max_concurrent=3) -> list[dict]` — apply an async function to every item with semaphore throttling and per-item error envelopes.

**Why it matters:** One failed LLM call should not abort 99 successful ones. `process_batch` catches exceptions inside each coroutine and packages them as `{item, status:'error', result:None, error:str}` dicts instead of propagating them to `asyncio.gather`.

## Provided: asyncio (already imported) + throttled_gather

In [ ]:
import asyncio

async def throttled_gather(coros: list, max_concurrent: int) -> list:
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(coro):
        async with sem:
            return await coro
    return list(await asyncio.gather(*[_run(c) for c in coros]))

## Your Implementation

In [ ]:
async def process_batch(
    items: list, async_fn, max_concurrent: int = 3
) -> list[dict]:
    """
    Batch-process items with async_fn; wrap each result in an error envelope.

    Args:
        items:          List of items to process.
        async_fn:       Async function: async_fn(item) -> result.
        max_concurrent: Max simultaneous coroutines (default 3).

    Returns:
        list[dict] — one per item, always len(items) records:
        On success: {'item': item, 'status': 'ok', 'result': val, 'error': None}
        On error:   {'item': item, 'status': 'error', 'result': None, 'error': str}
    """
    # TODO: sem = asyncio.Semaphore(max_concurrent)
    # TODO: async def _run(item):
    #     async with sem:
    #         try:
    #             result = await async_fn(item)
    #             return {'item': item, 'status': 'ok', 'result': result, 'error': None}
    #         except Exception as e:
    #             return {'item': item, 'status': 'error', 'result': None, 'error': str(e)}
    # TODO: return list(await asyncio.gather(*[_run(i) for i in items]))
    pass

## Check Your Work

In [ ]:
import asyncio

async def _run_checks():
    total = 5
    passed = 0

    # Mock async functions for testing (no LLM calls)
    async def _good(item):
        await asyncio.sleep(0)
        return f'done:{item}'

    async def _fail_on_bad(item):
        await asyncio.sleep(0)
        if item == 'bad':
            raise ValueError('intentional failure')
        return f'ok:{item}'

    # Check 1: defined
    try:
        assert 'process_batch' in globals()
        passed += 1; print('\u2705 Check 1: process_batch defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    # Check 2: result list has correct keys
    try:
        results = await process_batch(['a', 'b'], _good)
        assert isinstance(results, list)
        for r in results:
            for key in ('item', 'status', 'result', 'error'):
                assert key in r, f'missing key {key!r} in {r}'
        passed += 1; print('\u2705 Check 2: result dicts have item, status, result, error')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: successful items have status='ok' and non-None result
    try:
        results = await process_batch(['x', 'y', 'z'], _good)
        for r in results:
            assert r['status'] == 'ok', f"status should be 'ok': {r}"
            assert r['result'] is not None, f'result should not be None: {r}'
            assert r['error'] is None, f"error should be None for ok: {r}"
        passed += 1; print('\u2705 Check 3: ok items have status=ok, non-None result, error=None')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: failing items have status='error', result=None, error=str
    try:
        results = await process_batch(['good', 'bad', 'also_good'], _fail_on_bad)
        ok_items  = [r for r in results if r['status'] == 'ok']
        err_items = [r for r in results if r['status'] == 'error']
        assert len(ok_items)  == 2, f'expected 2 ok, got {len(ok_items)}'
        assert len(err_items) == 1, f'expected 1 error, got {len(err_items)}'
        err = err_items[0]
        assert err['item']   == 'bad',  f"error item should be 'bad': {err}"
        assert err['result'] is None,   f'error result should be None: {err}'
        assert isinstance(err['error'], str) and err['error'], \
            f'error should be non-empty str: {err}'
        passed += 1; print('\u2705 Check 4: failing items captured as error envelopes')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: result length always == len(items)
    try:
        for n in (0, 1, 5):
            results = await process_batch(list(range(n)), _good)
            assert len(results) == n, \
                f'n={n}: expected {n} results, got {len(results)}'
        passed += 1; print('\u2705 Check 5: result length always equals len(items)')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


await _run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
async def process_batch(
    items: list, async_fn, max_concurrent: int = 3
) -> list[dict]:
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(item):
        async with sem:
            try:
                result = await async_fn(item)
                return {"item": item, "status": "ok",
                        "result": result, "error": None}
            except Exception as e:
                return {"item": item, "status": "error",
                        "result": None, "error": str(e)}
    return list(await asyncio.gather(*[_run(i) for i in items]))
```

</details>